# VitroVision — สร้าง Seed จาก SAM3 (30 ภาพ annotate)

รันบน **Colab GPU** (ต้อง token ที่เข้าถึง `facebook/sam3` — gated).

### 📥 เตรียมก่อน (บน Google Drive ของคุณ) สร้างโฟลเดอร์ `VitroVision_colab/` แล้ววาง:
- `seed_images/` — ภาพ 30 ภาพที่คัดแล้ว (`001.jpg` ... `030.jpg`) จาก `data/work/annotate/images/` ของโปรเจกต์
- `sam3_growth_pipeline.py` — คัดจาก `src/sam3_growth_pipeline.py`
- `train_unet_distill.py` — คัดจาก `src/train_unet_distill.py`

> วิธีปลด `annotate` คัด images ออกมา: อัปโหลดโฟลเดอร์ `annotate/images` แล้วเปลี่ยนชื่อเป็น `seed_images` (หรือ zip ขึ้นไปก็ได้)


In [ ]:
!pip -q install torch torchvision transformers opencv-python pillow matplotlib pandas numpy huggingface_hub tabulate
print('deps OK')


In [ ]:
# วาง Hugging Face token ที่นี่ (huggingface.co -> Settings -> Access Tokens)
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxx"   # <- วาง token ของคุณ (สิทธิ์อ่าน facebook/sam3)

import os
from huggingface_hub import login
tok = os.environ.get('HF_TOKEN') or HF_TOKEN
if not tok or not tok.startswith('hf_'):
    raise SystemExit('ยังไม่ได้วาง token - แก้ HF_TOKEN = \'hf...\' ใน cell นี้ แล้วรันใหม่')
login(token=tok, add_to_git_credential=False)
os.environ['HF_TOKEN'] = tok
print('HF login OK')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')


In [ ]:
import os, shutil, glob, zipfile
DRIVE = '/content/drive/MyDrive'
SRC = os.path.join(DRIVE, 'VitroVision_colab')
WORK = '/content/seed_work'
os.makedirs(os.path.join(WORK, 'images'), exist_ok=True)

# 1) copy 2 source scripts to /content (must be same dir for imports)
for f in ('sam3_growth_pipeline.py', 'train_unet_distill.py'):
    sp = os.path.join(SRC, f)
    assert os.path.exists(sp), f'Missing {f} in {SRC} -> upload it'
    shutil.copy(sp, '/content/')
print('scripts:', os.path.exists('/content/sam3_growth_pipeline.py'), os.path.exists('/content/train_unet_distill.py'))

# 2) find ALL jpg (any structure: flat folder / subfolder / inside a zip)
def find_jpg(root):
    out = []
    for r, ds, fs in os.walk(root):
        ds[:] = [d for d in ds if not d.startswith('.')]
        for fn in fs:
            if fn.lower().endswith('.jpg'):
                out.append(os.path.join(r, fn))
    return out

jpg = find_jpg(SRC)
print('jpg under SRC (folder):', len(jpg))

if not jpg:
    # try the zip (support both 'seed_images.zip' and any zip in VitroVision_colab)
    for cand in glob.glob(os.path.join(SRC, 'seed_images*.zip')) + glob.glob(os.path.join(SRC, '*.zip')):
        print('extract:', cand)
        with zipfile.ZipFile(cand) as z:
            z.extractall(WORK)
        jpg = find_jpg(WORK)
        if jpg:
            break
    print('jpg after zip extract:', len(jpg))

assert jpg, 'ไม่พบภาพ - ตรวจว่า seed_images/ (โฟลเดอร์) หรือ seed_images.zip อยู่ใน MyDrive/VitroVision_colab/'

# 3) flat-copy into /content/seed_work/images (dedupe by filename)
flat = os.path.join(WORK, 'images')
seen = set()
for p in sorted(jpg):
    name = os.path.basename(p)
    if name not in seen:
        shutil.copy2(p, os.path.join(flat, name))
        seen.add(name)
imgs = sorted(f for f in os.listdir(flat) if f.lower().endswith('.jpg'))
print('images flat:', len(imgs), '| first:', imgs[:3])
assert len(imgs) >= 2, 'ไม่พอภาพ - ตรวจ folder/zip ให้ถูก'

In [ ]:
import subprocess
cmd = ['python', '/content/train_unet_distill.py', 'generate-pseudo',
       '--data', '/content/seed_work/images',
       '--out', '/content/seed_work/seed',
       '--hf-token', tok,
       '--limit', '30']
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-4000:])
print('--- stderr ---')
print(r.stderr[-2000:])
print('returncode:', r.returncode)


In [ ]:
import os, shutil
from datetime import datetime
seed_dir = '/content/seed_work/seed/pseudo_masks'
assert os.path.isdir(seed_dir), 'pseudo_masks not found - check previous cell'
stamp = datetime.now().strftime('%Y%m%d_%H%M')
base = f'/content/seed_masks_{stamp}'
shutil.make_archive(base, 'zip', seed_dir)
dst = f'/content/drive/MyDrive/VitroVision_colab/seed_masks_{stamp}.zip'
shutil.copy(base + '.zip', dst)
print('masks:', len(os.listdir(seed_dir)), '| saved Drive:', dst)
from google.colab import files
files.download(base + '.zip')


## 📥 หลังดาวน์โหลด

แตก zip กลับให้ได้โฟลเดอร์ `pseudo_masks/` แล้ววางที่ `data/work/annotate/seed/pseudo_masks/` ของโปรเจกต์
จากนั้นเปิด annotation tool:
```bash
python src/annotation_tool.py --data data/work/annotate/images \
  --seed data/work/annotate/seed/pseudo_masks \
  --out data/processed/ground_truth_masks --port 5000
```
